In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import struct

In [2]:
#custom functions to read ubyte files
def load_mnist_images(filepath):
    """Reads MNIST ubyte image files."""
    with open(filepath, 'rb') as f:
        magic, num_images, rows, cols = struct.unpack('>IIII', f.read(16))
        images = np.fromfile(f, dtype=np.uint8)
        return images.reshape(num_images, rows * cols)

def load_mnist_labels(filepath):
    """Reads MNIST ubyte label files."""
    with open(filepath, 'rb') as f:
        magic, num_labels = struct.unpack('>II', f.read(8))
        labels = np.fromfile(f, dtype=np.uint8)
        return labels

In [3]:
#Load data
X_train_np = load_mnist_images(r"C:/Users/USER/Documents/Devdocs/AI&ML/train-images-idx3-ubyte/train-images-idx3-ubyte")
y_train_np = load_mnist_labels(r"C:/Users/USER/Documents/Devdocs/AI&ML/train-labels-idx1-ubyte/train-labels-idx1-ubyte")
X_test_np = load_mnist_images(r"C:/Users/USER/Documents/Devdocs/AI&ML/t10k-images-idx3-ubyte/t10k-images-idx3-ubyte")
y_test_np = load_mnist_labels(r"C:/Users/USER/Documents/Devdocs/AI&ML/t10k-labels-idx1-ubyte\t10k-labels-idx1-ubyte")

In [4]:
#Preprocess and Convert to PyTorch Tensors
#Scale pixel intensities to [0, 1] and convert to 32-bit floats
X_train = torch.tensor(X_train_np / 255.0, dtype=torch.float32)
X_test = torch.tensor(X_test_np / 255.0, dtype=torch.float32)

# Labels need to be 64-bit integers (LongTensors) for PyTorch's CrossEntropyLoss
y_train = torch.tensor(y_train_np, dtype=torch.long)
y_test = torch.tensor(y_test_np, dtype=torch.long)

#Create DataLoaders
#DataLoaders help us iterate through the data automatically, in batches
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

batch_size = 200
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [5]:
#PyTorch Model
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        # Define the layers matching our previous scikit-learn architecture
        self.layers = nn.Sequential(
            nn.Linear(784, 128),  # Input layer (784) -> Hidden layer 1 (128)
            nn.ReLU(),            # Activation function
            nn.Linear(128, 64),   # Hidden layer 1 (128) -> Hidden layer 2 (64)
            nn.ReLU(),            # Activation function
            nn.Linear(64, 10)     # Hidden layer 2 (64) -> Output layer (10 digits)
        )

    def forward(self, x):
        return self.layers(x)

In [6]:
#Initialize Model, Loss, and Optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on device: {device}")

model = MLP().to(device)

#CrossEntropyLoss automatically applies Softmax, so our output layer doesn't need it
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Training on device: cpu


In [7]:
#Training Loop
epochs = 20
print("\nStarting Training...")

for epoch in range(epochs):
    model.train() #Set model to training mode
    running_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        #Move data to the same device as the model (GPU/CPU)
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        #Forward pass: compute predictions
        outputs = model(batch_X)
        
        #Calculate the loss
        loss = criterion(outputs, batch_y)
        
        #Backward pass: compute gradients
        optimizer.zero_grad() # Clear old gradients
        loss.backward()       # Calculate new gradients
        
        #Optimization step: update weights
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")


Starting Training...
Epoch 1/20 - Loss: 0.4764
Epoch 2/20 - Loss: 0.1936
Epoch 3/20 - Loss: 0.1406
Epoch 4/20 - Loss: 0.1117
Epoch 5/20 - Loss: 0.0893
Epoch 6/20 - Loss: 0.0745
Epoch 7/20 - Loss: 0.0623
Epoch 8/20 - Loss: 0.0533
Epoch 9/20 - Loss: 0.0458
Epoch 10/20 - Loss: 0.0399
Epoch 11/20 - Loss: 0.0324
Epoch 12/20 - Loss: 0.0282
Epoch 13/20 - Loss: 0.0242
Epoch 14/20 - Loss: 0.0216
Epoch 15/20 - Loss: 0.0184
Epoch 16/20 - Loss: 0.0144
Epoch 17/20 - Loss: 0.0133
Epoch 18/20 - Loss: 0.0123
Epoch 19/20 - Loss: 0.0093
Epoch 20/20 - Loss: 0.0073


In [8]:
#Evaluation
print("\nEvaluating the Model...")
model.eval() #Set model to evaluation mode (disables dropout, etc. if we had them)
correct = 0
total = 0

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = model(batch_X)
        
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

accuracy = (correct / total)*100
print(f"\nTest Accuracy: {accuracy:.2f}%")


Evaluating the Model...

Test Accuracy: 97.71%
